# Osoba B: GPT-2 + PEFT

Zakres (Część I, ~2,75 pkt): I.B, I.C, I.D, pkt 3 i 5 z [instrukcji](../Zadanie_2_zmodyfikowane.md).

## Do zrobienia

1. **I.B**: zapoznać się z biblioteką PEFT (quicktour + przykłady). Krótka notatka dla osoby A: jaki adapter (np. LoRA), jakie parametry, `task_type`.
2. **I.C**: powtórzyć tutorial z Medium (PEFT BERT-base, diagnozy medyczne) we własnym środowisku.
3. **I.D**: zapoznać się z klasami `GPT2Model`, `GPT2LMHeadModel`, `GPT2ForSequenceClassification`, `GPT2ForTokenClassification` (notatka w notebooku).
4. **Pkt 3: powtórzyć pkt 1 i 2 dla GPT-2**, model `sdadas/polish-gpt2-medium`, z adapterem PEFT:
   - klasyfikacja tekstu (`GPT2ForSequenceClassification`) i tokenów (`GPT2ForTokenClassification`), ocena na zbiorze testowym,
   - wizualizacja przestrzeni wektorowej dla tekstów testowych (tylko klasyfikacja tekstu), model bazowy vs dostrojony, dwa interaktywne wykresy + opis różnic.
5. **Pkt 5: augmentacja generacją GPT-2** (temperatura, top-p, top-k): porównać kilka ustawień, jaka jest jakość i różnorodność tekstów. Potem trening na zbiorze powiększonym.

## Ustalone (nie zmieniamy bez uzgodnienia z A)

- Model: `sdadas/polish-gpt2-medium`.
- Etykiety tekstu: `plus_m, minus_m, zero, amb` (`LABELS`, `LABEL2ID`). Tagi tokenów: BIO `wzmocnienie / oslabienie / negacja` (`TAGS`, `TAG2ID`).
- **Reprezentacja tekstu = ostatni token sekwencji** (ostatni token, który nie jest paddingiem), nie `[CLS]`. Dotyczy wizualizacji.
- **Maskowanie `-100`**: w klasyfikacji tokenów padding i znacznik `<eos>` dostają etykietę `-100`.
- Zbiór testowy jest wspólny z osobą A i nigdy nie jest augmentowany:
  - tekst: `load_text('test')`; zbiór treningowy: `load_text('train')`, walidacyjny: `load_text('dev')`,
  - tokeny: `split_tokens(load_tokens())` (seed 42).
- Augmentacja dotyczy tylko zbioru treningowego. Wynik zapisujemy w `../data/aug_gpt2.jsonl` (pola `text`, `label`, gdzie `label` to nazwa etykiety, a nie liczba). Finalny model trenujemy na `train + aug`.
- Adaptery i checkpointy zapisujemy w `../artifacts/`.

## Części wspólne (`../utils/`)

| Co | Skąd |
|---|---|
| wczytanie tekstów, etykiety | `utils.data`: `load_text`, `LABELS`, `LABEL2ID` |
| tagi BIO i podział train/test dla tokenów | `utils.tokens`: `load_tokens`, `split_tokens`, `TAGS`, `TAG2ID` |
| tagi słów → subtokeny (`-100`) | `utils.align`: `encode_tokens` |
| wykres przestrzeni wektorowej | `utils.plots`: `plot_embeddings` |

## Uwagi

- Tokenizer GPT-2: `AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)` (wymagane przy `is_split_into_words=True`) oraz `tokenizer.pad_token = tokenizer.eos_token` i `model.config.pad_token_id = tokenizer.pad_token_id`.
- `encode_tokens` oznacza `-100` tylko subtokeny niebędące pierwszym w słowie i pozycje specjalne. Jeśli tokenizer dopisuje `<eos>`, sprawdź, że dostał `-100` (`word_ids()` zwraca dla niego `None`).
- Wizualizacja: ostatni nie-paddingowy token (`attention_mask.sum(1) - 1`) z ostatniej warstwy ukrytej, przed głowicą. Dla `padding_side='right'` pamiętaj o tym indeksie.
- **Klasyfikacja tokenów ma mało danych**: anotacje z Zadania 1 to tylko 35 tekstów. Opisz to w wnioskach; rozszerzenie ustalamy z zespołem.
- Augmentacja: etykietę zachowaj przez prompt zawierający próbkę/początek tekstu z danej klasy. Odrzuć za krótkie i zduplikowane teksty.
- Metryki: tekst → accuracy i macro-F1 (`sklearn`); tokeny → precision/recall/F1 per tag, z pominięciem `-100`.


In [ ]:
import sys

sys.path.insert(0, "..")
from utils.align import encode_tokens
from utils.data import LABEL2ID, LABELS, load_text
from utils.plots import plot_embeddings
from utils.tokens import TAG2ID, TAGS, load_tokens, split_tokens

MODEL = "sdadas/polish-gpt2-medium"
train, dev, test = load_text("train"), load_text("dev"), load_text("test")
tok_train, tok_test = split_tokens(load_tokens())
len(train), len(dev), len(test), len(tok_train), len(tok_test)

## I.B, I.C: PEFT (notatka + powtórzony tutorial)

## I.D: klasy GPT-2 (notatka)

## 3a. Klasyfikacja tekstu + PEFT

## 3b. Klasyfikacja tokenów + PEFT (padding i `<eos>` → `-100`)

## 3c. Wizualizacja przestrzeni wektorowej (ostatni token): przed i po strojeniu

## 5. Augmentacja generacją (temperatura, top-p, top-k) i trening na zbiorze powiększonym